In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "modeling_datasets" / "final").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/modeling_datasets/final")


PROJECT_ROOT = resolve_project_root()
FINAL_DIR = PROJECT_ROOT / "data" / "modeling_datasets" / "final"
COLLEGE_DIR = PROJECT_ROOT / "data" / "modeling_datasets" / "college" / "aggregated_stats"
OUTPUT_PATH = FINAL_DIR / "factoids" / "gridiron_scouting_factoids.csv"

SUCCESS_THRESHOLDS = {
    "contributor": 20.0,
    "multi-year starter": 50.0,
    "elite": 80.0,
}

THRESHOLD_METRICS = [
    ("contributor", "is_contributor", "contributor rate (>=20)"),
    ("multi-year starter", "is_multi_year_starter", "multi-year starter rate (>=50)"),
    ("elite", "is_elite", "elite rate (>=80)"),
]

STAR_BANDS = [
    ("5-star", 0.9834, np.inf),
    ("4-star", 0.8900, 0.9834),
    ("3-star", 0.7970, 0.8900),
    ("2-star", -np.inf, 0.7970),
]

POSITION_BIOMETRIC_MARKS = {
    "QB": {"height_inches": 76, "weight_lbs": 220},
    "RB": {"height_inches": 71, "weight_lbs": 205},
    "WR": {"height_inches": 74, "weight_lbs": 200},
    "TE": {"height_inches": 76, "weight_lbs": 240},
    "OL": {"height_inches": 77, "weight_lbs": 310},
    "EDGE": {"height_inches": 75, "weight_lbs": 250},
    "IDL": {"height_inches": 75, "weight_lbs": 300},
    "LB": {"height_inches": 73, "weight_lbs": 230},
    "DB": {"height_inches": 72, "weight_lbs": 190},
    "SPEC": {"height_inches": 72, "weight_lbs": 190},
}

MIN_SAMPLE = 30
MIN_RELATIVE_DIFF_PCT = 15.0
TOP_SKILL_COUNT = 3
TOP_STATE_COUNT = 12


def pct_diff_from_baseline(value: float, baseline: float) -> float:
    if pd.isna(value) or pd.isna(baseline):
        return np.nan
    if baseline == 0:
        if value == 0:
            return 0.0
        return np.nan
    return ((value - baseline) / abs(baseline)) * 100.0


def direction_word(delta_pct: float) -> str:
    return "higher" if delta_pct >= 0 else "lower"


def format_metric_value(metric_name: str, value: float) -> str:
    if "rate" in metric_name.lower() or "probability" in metric_name.lower():
        return f"{value * 100:.1f}%"
    return f"{value:.2f}"


def normalize_tags(tags: list) -> str:
    normalized = []
    for tag in tags:
        tag_clean = str(tag).strip().lower().replace(" ", "_")
        if tag_clean:
            normalized.append(tag_clean)
    return "|".join(sorted(set(normalized)))


def add_factoid(factoids: list, position: str, factoid_type: str, criteria: str, metric_name: str,
                value: float, baseline: float, n: int, tags: list):
    delta_pct = pct_diff_from_baseline(value, baseline)
    if pd.isna(delta_pct):
        return
    if n < MIN_SAMPLE:
        return
    if abs(delta_pct) < MIN_RELATIVE_DIFF_PCT:
        return

    metric_value_txt = format_metric_value(metric_name, value)
    text = (
        f"[{position}] Insight: {position} recruits who {criteria} saw an average {metric_name} "
        f"of {metric_value_txt}, which is {abs(delta_pct):.1f}% {direction_word(delta_pct)} "
        f"than the position average."
    )
    factoids.append({
        "position": position,
        "factoid_type": factoid_type,
        "factoid_text": text,
        "tags": normalize_tags(tags),
        "n": n,
        "delta_pct": delta_pct,
    })


def add_threshold_insights(
    factoids: list,
    position: str,
    factoid_type: str,
    criteria: str,
    subset_df: pd.DataFrame,
    baselines: dict,
    n: int,
    tags: list,
):
    for threshold_label, rate_col, metric_name in THRESHOLD_METRICS:
        add_factoid(
            factoids=factoids,
            position=position,
            factoid_type=factoid_type,
            criteria=criteria,
            metric_name=metric_name,
            value=float(subset_df[rate_col].mean()),
            baseline=baselines[f"{threshold_label}_rate"],
            n=n,
            tags=tags + ["threshold", f"ge_{int(SUCCESS_THRESHOLDS[threshold_label])}", threshold_label],
        )


def parse_position(file_name: str) -> str:
    return file_name.split("_")[0].upper()


def load_position_data(position: str) -> pd.DataFrame:
    final_file = FINAL_DIR / f"{position}_modeling_with_target.csv"
    college_file = COLLEGE_DIR / f"{position}_college_player_career_ready.csv"

    if not final_file.exists() or not college_file.exists():
        return pd.DataFrame()

    df_final = pd.read_csv(final_file)
    df_college = pd.read_csv(college_file)

    if "sports_ref_id" not in df_final.columns or "Player_ID" not in df_college.columns:
        return pd.DataFrame()

    df_final["sports_ref_id"] = df_final["sports_ref_id"].astype(str).str.strip()
    df_college["Player_ID"] = df_college["Player_ID"].astype(str).str.strip()

    df_join = df_final.merge(
        df_college,
        how="inner",
        left_on="sports_ref_id",
        right_on="Player_ID",
        suffixes=("", "_college"),
    )

    if "target_career_score_0_100" not in df_join.columns:
        return pd.DataFrame()

    df_join["target_career_score_0_100"] = pd.to_numeric(df_join["target_career_score_0_100"], errors="coerce")
    df_join = df_join[df_join["target_career_score_0_100"].notna()].copy()

    if df_join.empty:
        return df_join

    df_join["is_contributor"] = (df_join["target_career_score_0_100"] >= SUCCESS_THRESHOLDS["contributor"]).astype(int)
    df_join["is_multi_year_starter"] = (df_join["target_career_score_0_100"] >= SUCCESS_THRESHOLDS["multi-year starter"]).astype(int)
    df_join["is_elite"] = (df_join["target_career_score_0_100"] >= SUCCESS_THRESHOLDS["elite"]).astype(int)

    return df_join


def metric_baselines(df_pos: pd.DataFrame) -> dict:
    return {
        "avg_score": float(df_pos["target_career_score_0_100"].mean()),
        "contributor_rate": float(df_pos["is_contributor"].mean()),
        "multi-year starter_rate": float(df_pos["is_multi_year_starter"].mean()),
        "elite_rate": float(df_pos["is_elite"].mean()),
    }


def skill_is_integer_series(series: pd.Series) -> pd.Series:
    vals = pd.to_numeric(series, errors="coerce")
    return vals.notna() & np.isclose(vals % 1, 0)


final_files = sorted(FINAL_DIR.glob("*_modeling_with_target.csv"))
positions = [parse_position(f.name) for f in final_files]

all_factoids = []
join_diagnostics = []

for position in positions:
    df_pos = load_position_data(position)

    join_diagnostics.append({
        "position": position,
        "joined_rows": len(df_pos),
    })

    if df_pos.empty:
        continue

    baselines = metric_baselines(df_pos)

    # 1) Biometric Thresholds
    marks = POSITION_BIOMETRIC_MARKS.get(position, {})
    for col_name, threshold in marks.items():
        if col_name not in df_pos.columns:
            continue

        metric_series = pd.to_numeric(df_pos[col_name], errors="coerce")
        valid = metric_series.notna()
        if valid.sum() < MIN_SAMPLE:
            continue

        over_mask = valid & (metric_series > threshold)
        under_mask = valid & (metric_series <= threshold)

        if over_mask.sum() >= MIN_SAMPLE:
            over_df = df_pos.loc[over_mask]
            add_factoid(
                all_factoids,
                position,
                "biometric_threshold",
                f"were above {col_name} {threshold}",
                "career success score",
                float(over_df["target_career_score_0_100"].mean()),
                baselines["avg_score"],
                int(over_mask.sum()),
                [position, "biometric_threshold", col_name, "above", str(threshold)],
            )
            add_threshold_insights(
                factoids=all_factoids,
                position=position,
                factoid_type="biometric_threshold",
                criteria=f"were above {col_name} {threshold}",
                subset_df=over_df,
                baselines=baselines,
                n=int(over_mask.sum()),
                tags=[position, "biometric_threshold", col_name, "above", str(threshold)],
            )

        if under_mask.sum() >= MIN_SAMPLE:
            under_df = df_pos.loc[under_mask]
            add_factoid(
                all_factoids,
                position,
                "biometric_threshold",
                f"were at or below {col_name} {threshold}",
                "career success score",
                float(under_df["target_career_score_0_100"].mean()),
                baselines["avg_score"],
                int(under_mask.sum()),
                [position, "biometric_threshold", col_name, "at_or_below", str(threshold)],
            )
            add_threshold_insights(
                factoids=all_factoids,
                position=position,
                factoid_type="biometric_threshold",
                criteria=f"were at or below {col_name} {threshold}",
                subset_df=under_df,
                baselines=baselines,
                n=int(under_mask.sum()),
                tags=[position, "biometric_threshold", col_name, "at_or_below", str(threshold)],
            )

    # 2) Skill Influence
    skill_cols = [c for c in df_pos.columns if c.startswith("skill_")]
    skill_corr_rows = []

    for skill_col in skill_cols:
        skill_vals = pd.to_numeric(df_pos[skill_col], errors="coerce")
        valid_int = skill_is_integer_series(df_pos[skill_col])

        if valid_int.sum() < MIN_SAMPLE:
            continue

        corr_df = pd.DataFrame({
            "skill": skill_vals[valid_int],
            "score": df_pos.loc[valid_int, "target_career_score_0_100"],
        }).dropna()

        if len(corr_df) < MIN_SAMPLE:
            continue
        if corr_df["skill"].nunique() < 2:
            continue

        corr_val = corr_df["skill"].corr(corr_df["score"])
        if pd.notna(corr_val):
            skill_corr_rows.append((skill_col, abs(float(corr_val))))

    top_skills = [s for s, _ in sorted(skill_corr_rows, key=lambda x: x[1], reverse=True)[:TOP_SKILL_COUNT]]

    for skill_col in top_skills:
        skill_vals = pd.to_numeric(df_pos[skill_col], errors="coerce")
        valid_int = skill_is_integer_series(df_pos[skill_col])
        high_skill_mask = valid_int & skill_vals.isin([9, 10])

        if high_skill_mask.sum() < MIN_SAMPLE:
            continue

        high_skill_df = df_pos.loc[high_skill_mask]
        add_factoid(
            all_factoids,
            position,
            "skill_influence",
            f"had {skill_col} ratings of 9 or 10",
            "career success score",
            float(high_skill_df["target_career_score_0_100"].mean()),
            baselines["avg_score"],
            int(high_skill_mask.sum()),
            [position, "skill_influence", skill_col, "rating_9_or_10"],
        )
        add_threshold_insights(
            factoids=all_factoids,
            position=position,
            factoid_type="skill_influence",
            criteria=f"had {skill_col} ratings of 9 or 10",
            subset_df=high_skill_df,
            baselines=baselines,
            n=int(high_skill_mask.sum()),
            tags=[position, "skill_influence", skill_col, "rating_9_or_10"],
        )

    # 3) Star Thresholds
    if "rating" in df_pos.columns:
        rating_vals = pd.to_numeric(df_pos["rating"], errors="coerce")
        for star_label, low, high in STAR_BANDS:
            if np.isinf(low):
                band_mask = rating_vals < high
            elif np.isinf(high):
                band_mask = rating_vals >= low
            else:
                band_mask = (rating_vals >= low) & (rating_vals < high)

            band_mask = band_mask & rating_vals.notna()
            n_band = int(band_mask.sum())
            if n_band < MIN_SAMPLE:
                continue

            band_df = df_pos.loc[band_mask]
            add_factoid(
                all_factoids,
                position,
                "star_threshold",
                f"were {star_label} prospects",
                "career success score",
                float(band_df["target_career_score_0_100"].mean()),
                baselines["avg_score"],
                n_band,
                [position, "star_threshold", star_label],
            )
            add_threshold_insights(
                factoids=all_factoids,
                position=position,
                factoid_type="star_threshold",
                criteria=f"were {star_label} prospects",
                subset_df=band_df,
                baselines=baselines,
                n=n_band,
                tags=[position, "star_threshold", star_label],
            )

    # 4) State Analysis
    if "state" in df_pos.columns:
        state_series = df_pos["state"].astype(str).str.strip().str.upper()
        valid_state_mask = state_series.notna() & (state_series != "") & (state_series != "NAN")

        if valid_state_mask.sum() >= MIN_SAMPLE:
            state_counts = state_series[valid_state_mask].value_counts()
            candidate_states = state_counts[state_counts >= MIN_SAMPLE].head(TOP_STATE_COUNT).index.tolist()

            for state_code in candidate_states:
                state_mask = valid_state_mask & (state_series == state_code)
                n_state = int(state_mask.sum())
                if n_state < MIN_SAMPLE:
                    continue

                state_df = df_pos.loc[state_mask]
                add_factoid(
                    all_factoids,
                    position,
                    "state_analysis",
                    f"were from {state_code}",
                    "career success score",
                    float(state_df["target_career_score_0_100"].mean()),
                    baselines["avg_score"],
                    n_state,
                    [position, "state_analysis", "state", state_code],
                )
                add_threshold_insights(
                    factoids=all_factoids,
                    position=position,
                    factoid_type="state_analysis",
                    criteria=f"were from {state_code}",
                    subset_df=state_df,
                    baselines=baselines,
                    n=n_state,
                    tags=[position, "state_analysis", "state", state_code],
                )

    # 5) Prestige Lift
    prestige_flags = [
        "flag_all_usa_first_team" ,
        "flag_all_usa_second_team",
        "flag_all_american_bowl",
        "flag_all_america_game",
        "flag_gatorade_poy",
        "flag_the_opening",
        "flag_maxpreps_all_american"
    ]

    for flag_col in prestige_flags:
        if flag_col not in df_pos.columns:
            continue

        flag_vals = pd.to_numeric(df_pos[flag_col], errors="coerce").fillna(0)
        has_flag = flag_vals > 0
        no_flag = flag_vals <= 0

        n_has = int(has_flag.sum())
        n_no = int(no_flag.sum())

        if n_has >= MIN_SAMPLE:
            has_df = df_pos.loc[has_flag]
            add_factoid(
                all_factoids,
                position,
                "prestige_lift",
                f"had {flag_col}",
                "career success score",
                float(has_df["target_career_score_0_100"].mean()),
                baselines["avg_score"],
                n_has,
                [position, "prestige_lift", flag_col, "has_flag"],
            )
            add_threshold_insights(
                factoids=all_factoids,
                position=position,
                factoid_type="prestige_lift",
                criteria=f"had {flag_col}",
                subset_df=has_df,
                baselines=baselines,
                n=n_has,
                tags=[position, "prestige_lift", flag_col, "has_flag"],
            )

        if n_no >= MIN_SAMPLE:
            no_df = df_pos.loc[no_flag]
            add_factoid(
                all_factoids,
                position,
                "prestige_lift",
                f"did not have {flag_col}",
                "career success score",
                float(no_df["target_career_score_0_100"].mean()),
                baselines["avg_score"],
                n_no,
                [position, "prestige_lift", flag_col, "no_flag"],
            )
            add_threshold_insights(
                factoids=all_factoids,
                position=position,
                factoid_type="prestige_lift",
                criteria=f"did not have {flag_col}",
                subset_df=no_df,
                baselines=baselines,
                n=n_no,
                tags=[position, "prestige_lift", flag_col, "no_flag"],
            )

factoids_df = pd.DataFrame(all_factoids)
if not factoids_df.empty:
    factoids_df = (
        factoids_df
        .sort_values(["position", "factoid_type", "delta_pct"], ascending=[True, True, False])
        .drop_duplicates(subset=["position", "factoid_type", "factoid_text", "tags"])
        .reset_index(drop=True)
    )

if factoids_df.empty:
    output_df = pd.DataFrame(columns=["position", "factoid_type", "factoid_text", "tags"])
else:
    output_df = factoids_df[["position", "factoid_type", "factoid_text", "tags"]]

output_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

join_diagnostics_df = pd.DataFrame(join_diagnostics).sort_values("position")
print(f"Saved scouting factoids: {OUTPUT_PATH}")
print(f"Total factoids: {len(output_df)}")

display(join_diagnostics_df)
display(output_df.head(20))

Saved scouting factoids: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\final\factoids\gridiron_scouting_factoids.csv
Total factoids: 336


,position,joined_rows
0,DB,4028
1,EDGE,827
2,IDL,1840
3,LB,2397
4,OL,4130
5,QB,1414
6,RB,1608
7,SPEC,570
8,TE,1191
9,WR,2858


,position,factoid_type,factoid_text,tags
0,DB,biometric_threshold,[DB] Insight: DB recruits who were above weigh...,190|above|biometric_threshold|db|elite|ge_80|t...
1,DB,prestige_lift,[DB] Insight: DB recruits who had flag_all_ame...,db|elite|flag_all_american_bowl|ge_80|has_flag...
2,DB,prestige_lift,[DB] Insight: DB recruits who had flag_all_ame...,db|elite|flag_all_america_game|ge_80|has_flag|...
3,DB,prestige_lift,[DB] Insight: DB recruits who had flag_the_ope...,db|elite|flag_the_opening|ge_80|has_flag|prest...
4,DB,prestige_lift,[DB] Insight: DB recruits who had flag_all_ame...,db|flag_all_america_game|ge_50|has_flag|multi-...
5,DB,prestige_lift,[DB] Insight: DB recruits who had flag_the_ope...,db|flag_the_opening|ge_50|has_flag|multi-year_...
6,DB,prestige_lift,[DB] Insight: DB recruits who had flag_all_ame...,db|flag_all_america_game|has_flag|prestige_lift
7,DB,prestige_lift,[DB] Insight: DB recruits who had flag_all_ame...,db|flag_all_american_bowl|ge_50|has_flag|multi...
8,DB,prestige_lift,[DB] Insight: DB recruits who had flag_the_ope...,db|flag_the_opening|has_flag|prestige_lift
9,DB,prestige_lift,[DB] Insight: DB recruits who had flag_all_ame...,db|flag_all_american_bowl|has_flag|prestige_lift


In [2]:
import json
import hashlib
from datetime import datetime, timezone

import torch
from sentence_transformers import SentenceTransformer

# Use in-memory output_df when available; otherwise fallback to saved CSV.
if "output_df" in globals() and isinstance(output_df, pd.DataFrame) and not output_df.empty:
    factoid_source_df = output_df.copy()
else:
    factoid_source_df = pd.read_csv(OUTPUT_PATH)

required_cols = ["position", "factoid_type", "factoid_text"]
for c in required_cols:
    if c not in factoid_source_df.columns:
        raise ValueError(f"Missing required column for embedding: {c}")

if "tags" not in factoid_source_df.columns:
    factoid_source_df["tags"] = (
        factoid_source_df["position"].astype(str).str.lower() + "|" +
        factoid_source_df["factoid_type"].astype(str).str.lower()
    )

factoid_source_df = factoid_source_df.dropna(subset=["factoid_text"]).copy()
factoid_source_df["factoid_text"] = factoid_source_df["factoid_text"].astype(str).str.strip()
factoid_source_df = factoid_source_df[factoid_source_df["factoid_text"] != ""].copy()

if factoid_source_df.empty:
    raise ValueError("No factoids available to embed.")

# Intel B580 / Intel Arc path: use torch XPU when available.
if hasattr(torch, "xpu") and torch.xpu.is_available():
    embedding_device = "xpu"
else:
    embedding_device = "cpu"

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name, device=embedding_device)

factoid_source_df["search_text"] = factoid_source_df.apply(
    lambda r: (
        f"Position: {r['position']}. "
        f"Factoid type: {r['factoid_type']}. "
        f"Tags: {r['tags']}. "
        f"Insight: {r['factoid_text']}"
    ),
    axis=1,
)

embeddings = embedder.encode(
    factoid_source_df["search_text"].tolist(),
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

utc_now = datetime.now(timezone.utc).isoformat()

supabase_rows = []
for idx, row in factoid_source_df.reset_index(drop=True).iterrows():
    emb = embeddings[idx].astype(float).tolist()
    uid_source = f"{row['position']}|{row['factoid_type']}|{row['factoid_text']}|{row['tags']}"
    uid = hashlib.sha1(uid_source.encode("utf-8")).hexdigest()

    supabase_rows.append({
        "id": uid,
        "position": row["position"],
        "factoid_type": row["factoid_type"],
        "factoid_text": row["factoid_text"],
        "tags": row["tags"],
        "search_text": row["search_text"],
        "embedding": json.dumps(emb),
        "embedding_model": embedding_model_name,
        "embedding_device": embedding_device,
        "created_at": utc_now,
    })

supabase_df = pd.DataFrame(supabase_rows)

supabase_csv_path = FINAL_DIR / "factoids" / "gridiron_scouting_factoids_supabase_vectors.csv"
supabase_jsonl_path = FINAL_DIR / "factoids" / "gridiron_scouting_factoids_supabase_vectors.jsonl"

# CSV for direct load tooling
supabase_df.to_csv(supabase_csv_path, index=False, encoding="utf-8-sig")

# JSONL for API upserts
with open(supabase_jsonl_path, "w", encoding="utf-8") as f:
    for rec in supabase_rows:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Embedding model: {embedding_model_name}")
print(f"Device used: {embedding_device}")
print(f"Rows embedded: {len(supabase_df)}")
print(f"Saved Supabase CSV: {supabase_csv_path}")
print(f"Saved Supabase JSONL: {supabase_jsonl_path}")

display(supabase_df[["id", "position", "factoid_type", "tags", "embedding_device"]].head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Device used: xpu
Rows embedded: 336
Saved Supabase CSV: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\final\factoids\gridiron_scouting_factoids_supabase_vectors.csv
Saved Supabase JSONL: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\final\factoids\gridiron_scouting_factoids_supabase_vectors.jsonl


,id,position,factoid_type,tags,embedding_device
0,93e45f3668644319ac6dcd27916d05e315d49e16,DB,biometric_threshold,190|above|biometric_threshold|db|elite|ge_80|t...,xpu
1,434f4ab76a49a4da8ac24457954e639bcc78989e,DB,prestige_lift,db|elite|flag_all_american_bowl|ge_80|has_flag...,xpu
2,07f5a33248827c9c640e9fa806dcacc2d61e759c,DB,prestige_lift,db|elite|flag_all_america_game|ge_80|has_flag|...,xpu
3,de3877fb52ee6db2e9373511678a98948232d4c3,DB,prestige_lift,db|elite|flag_the_opening|ge_80|has_flag|prest...,xpu
4,2c69b8ac1d4d174927307f6b7b923f863e47f2f2,DB,prestige_lift,db|flag_all_america_game|ge_50|has_flag|multi-...,xpu
5,27756db0d91e93df9f92a0f0e4273418bbb7ce27,DB,prestige_lift,db|flag_the_opening|ge_50|has_flag|multi-year_...,xpu
6,db53583afc79f96ad5e149a53221699814e91bfe,DB,prestige_lift,db|flag_all_america_game|has_flag|prestige_lift,xpu
7,cb188476acbe0203a2a9ddb678364275dc403ca4,DB,prestige_lift,db|flag_all_american_bowl|ge_50|has_flag|multi...,xpu
8,42d27135dece22574e67aaae082019bbc9ba1095,DB,prestige_lift,db|flag_the_opening|has_flag|prestige_lift,xpu
9,6a1e13d3079de3baa6a495fbb8cd7f70bff078f2,DB,prestige_lift,db|flag_all_american_bowl|has_flag|prestige_lift,xpu
